# Vehicle Collision Detector — Judge Testing Notebook

This notebook loads the pre-trained model from the **pickle checkpoint** and runs the full interactive judge testing system.

**Instructions:**
1. Run all cells in order
2. pip installs can be commented out if not needed
3. Change file paths if download is different
4. When prompted, enter a camera number (1–30) and an image number (1–30) to test

In [ ]:
# ── Cell 1: Download pickle from public link ─────────────────────
!pip install gdown -q

import gdown

FILE_ID     = "1jc3iPMAJveEG4pjX1TzFHcVlrXdTl9cQ"
PICKLE_PATH = "accident_classifier_checkpoint.pkl"

gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", PICKLE_PATH, quiet=False)
print("✅ Pickle downloaded successfully.")

In [ ]:
# ── Cell 2: Install / import dependencies ─────────────────────────────────────
# uncomment if pip installs are required
!pip install torch torchvision Pillow

import pickle
import torch
import torch.nn as nn
import requests
from PIL import Image
from io import BytesIO
from torchvision import transforms, models

print("✅ All dependencies loaded.")

In [ ]:
# ── Cell 3: Configuration ─────────────────────────
PICKLE_PATH = "accident_classifier_checkpoint.pkl" #change if local path is different
DEVICE      = torch.device("cpu")

In [ ]:
# ── Cell 4: Load model from pickle ────────────────────────────────────────────
def build_model(num_classes=2):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

with open(PICKLE_PATH, "rb") as f:
    checkpoint = pickle.load(f)

class_names = checkpoint["class_names"]
IMG_SIZE    = checkpoint["img_size"]

model = build_model(num_classes=len(class_names))
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"✅ Model loaded successfully from: {PICKLE_PATH}")
print(f"   Classes  : {class_names}")
print(f"   Img size : {IMG_SIZE}x{IMG_SIZE}")

In [ ]:
# ── Cell 5: Transforms (must match training) ──────────────────────────────────
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# ── Cell 6: Test image bank (15 accident + 15 normal) ─────────────────────────
test_image_bank = {
    # ACCIDENT IMAGES (A)
    1:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/01A.jpg",  "A"),
    2:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/02A.jpg",  "A"),
    3:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/03A.jpg",  "A"),
    4:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/04A.jpg",  "A"),
    5:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/05A.jpg",  "A"),
    6:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/06A.jpg",  "A"),
    7:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/07A.jpg",  "A"),
    8:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/08A.jpg",  "A"),
    9:  ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/09A.jpg",  "A"),
    10: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/10A.jpg", "A"),
    11: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/11A.jpg", "A"),
    12: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/12A.jpg", "A"),
    13: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/13A.jpg", "A"),
    14: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/14A.jpg", "A"),
    15: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/015A.jpg", "A"),
    # NO ACCIDENT IMAGES (NA)
    16: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/01N.jpg",  "NA"),
    17: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/02N.jpg",  "NA"),
    18: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/03N.jpg",  "NA"),
    19: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/04N.jpg",  "NA"),
    20: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/05N.jpg",  "NA"),
    21: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/06N.jpg",  "NA"),
    22: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/07N.jpg",  "NA"),
    23: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/08N.jpg",  "NA"),
    24: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/09N.jpg",  "NA"),
    25: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/10N.jpg", "NA"),
    26: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/11N.jpg", "NA"),
    27: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/12N.jpg", "NA"),
    28: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/13N.jpg", "NA"),
    29: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/14N.jpg", "NA"),
    30: ("https://raw.githubusercontent.com/Swetha-aaa/traffic-test-images/main/15N.jpg", "NA"),
}
print("✅ Test image bank ready.")

In [ ]:
# ── Cell 7: Helper functions ───────────────────────────────────────────────────

def fetch_lta_cameras():
    url = "https://api.data.gov.sg/v1/transport/traffic-images"
    try:
        response = requests.get(url)
        data = response.json()
        cameras_raw = data["items"][0]["cameras"]
        cameras = {}
        for i, cam in enumerate(cameras_raw[:30], start=1):
            cam_id    = cam["camera_id"]
            lat       = cam["location"]["latitude"]
            lon       = cam["location"]["longitude"]
            image_url = cam["image"]
            cameras[i] = {
                "id"   : cam_id,
                "lat"  : lat,
                "lon"  : lon,
                "image": image_url,
                "label": f"Camera {cam_id} (Lat: {lat:.4f}, Lon: {lon:.4f})"
            }
        print("✅ LTA cameras loaded successfully.")
        return cameras
    except Exception as e:
        print(f"❌ Failed to fetch cameras: {e}")
        return {}


def show_cameras(cameras):
    print("═" * 60)
    print("  LTA TRAFFIC CAMERA LOCATIONS")
    print("═" * 60)
    for num, cam in cameras.items():
        print(f"  {num:>2}. {cam['label']}")
    print("═" * 60)


def show_image_bank():
    print("═" * 60)
    print("  TEST IMAGE BANK")
    print("  A = Accident  |  NA = No Accident")
    print("═" * 60)
    for num, (path, label) in test_image_bank.items():
        tag = "🚨 Accident   " if label == "A" else "✅ No Accident"
        print(f"  Image {num:>2} : {tag}")
    print("═" * 60)


def run_prediction(image_path):
    if image_path.startswith("http"):
        response = requests.get(image_path)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        img = Image.open(image_path).convert("RGB")

    img_tensor = val_transforms(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        acc_prob = torch.softmax(model(img_tensor), dim=1)[0]
    accident_pred = class_names[acc_prob.argmax()]
    accident_conf = acc_prob.max().item()

    return accident_pred, accident_conf, acc_prob


print("✅ Helper functions defined.")

In [ ]:
# ── Cell 8: RUN JUDGE SYSTEM ──────────────────────────────────────────────────
# Run this cell to start the interactive judge testing session.
# You will be prompted to enter a camera number and an image number.

def run_judge_system():
    # Step 1 — fetch and show cameras
    print("\nFetching LTA cameras...\n")
    cameras = fetch_lta_cameras()
    if not cameras:
        print("Could not load cameras. Check internet connection.")
        return

    show_cameras(cameras)
    cam_choice = int(input("\nEnter camera number (1-30): "))
    if cam_choice not in cameras:
        print("Invalid camera number.")
        return
    selected_camera = cameras[cam_choice]
    print(f"\n✅ Camera selected : {selected_camera['label']}")
    print(f"   Camera ID       : {selected_camera['id']}")
    print(f"   Location        : Lat {selected_camera['lat']}, Lon {selected_camera['lon']}")

    # Step 2 — show image bank and let judge choose
    print()
    show_image_bank()
    img_choice = int(input("\nEnter image number to test (1-30): "))
    if img_choice not in test_image_bank:
        print("Invalid image number.")
        return

    image_path, true_label = test_image_bank[img_choice]
    true_label_text = "Accident" if true_label == "A" else "No Accident"
    print(f"\n✅ Image {img_choice} selected  |  True label: {true_label_text}")

    # Step 3 — run model prediction
    print("\nRunning model prediction...\n")
    accident_pred, accident_conf, acc_prob = run_prediction(image_path)

    # Step 4 — output results
    print("═" * 60)
    print(f"  CAMERA LOCATION    : {selected_camera['label']}")
    print(f"  CAMERA ID          : {selected_camera['id']}")
    print(f"  IMAGE TESTED       : Image {img_choice}  [{true_label_text}]")
    print("─" * 60)
    print("  COLLISION DETECTION RESULTS")
    print("─" * 60)
    for cls, p in zip(class_names, acc_prob):
        bar = "█" * int(p.item() * 20)
        print(f"  {cls:<12} : {p:.2%}  {bar}")
    print("─" * 60)
    if accident_pred == "accident":
        print(f"  🚨 COLLISION DETECTED at {selected_camera['label']}")
        print(f"  CONFIDENCE         : {accident_conf:.2%}")
    else:
        print(f"  ✅ NO COLLISION at {selected_camera['label']}")
        print(f"  CONFIDENCE         : {accident_conf:.2%}")
    print("─" * 60)

    # Step 5 — correctness check
    predicted_label = "A" if accident_pred == "accident" else "NA"
    if predicted_label == true_label:
        print(f"  VERDICT : ✅ CORRECT  (Model predicted: {accident_pred.upper()})")
    else:
        print(f"  VERDICT : ❌ INCORRECT (True: {true_label_text} | Predicted: {accident_pred.upper()})")
    print("═" * 60)


run_judge_system()